In [ ]:
import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "transformers", "scikit-learn"])
%pip install einops

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
import sys, os, random, time, json, copy, warnings, math
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef, accuracy_score, recall_score
from transformers import AutoModel

warnings.filterwarnings("ignore", message="enable_nested_tensor")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 5090


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

In [ ]:
CONFIG = {
    # -- paths --
    "data_dir": "/Data",
    "imbalanced_file": "/Data/Imbalanced_Independent.txt",

    # -- PLM --
    "plm_name": "Synthyra/ESMplusplus_large",
    "plm_dim": 1152,
    "seq_len": 41,
    "center_idx": 20,

    # -- branch A: transformer head --
    "proj_dim": 256,
    "n_transformer_layers": 2,
    "n_heads": 4,
    "ff_mult": 4,
    "trans_dropout": 0.3,

    # -- branch B: CNN-BiGRU --
    "seq_feat_dim": 40,
    "conv_kernels": [1, 2, 3, 4, 5, 6],
    "conv_channels": 32,
    "gru_hidden": 128,
    "seq_dropout": 0.3,

    # -- fusion + classifier --
    "branch_a_out": 256,
    "branch_b_out": 64,
    "fused_hidden": 128,
    "clf_dropout": 0.3,

    # -- training (identical to v5) --
    "lr": 3e-4,
    "weight_decay": 5e-4,
    "batch_size": 64,
    "max_epochs": 100,
    "patience": 10,
    "grad_clip": 1.0,
    "threshold": 0.5,
    "n_folds": 5,
    "seeds": [42, 43, 44, 45, 46, 47, 48],

    # -- training enhancements (identical to v5) --
    "mixup_alpha": 0.2,
    "label_smoothing": 0.1,
    "swa_epochs": 10,
    "swa_lr_factor": 0.2,

    # -- NEW: threshold optimization --
    # Set to True to search for a threshold that maximizes accuracy
    # without dropping MCC below the 0.5-threshold MCC.
    # Set to False to use fixed 0.5 threshold (same as v5).
    "optimize_threshold": True,

    # -- output --
    "results_dir": "./results_v5_1_final",
}
os.makedirs(CONFIG["results_dir"], exist_ok=True)
print("Config ready.")
print(f"  Seeds: {CONFIG['seeds']} ({len(CONFIG['seeds'])} seeds)")
print(f"  Total models: {len(CONFIG["seeds"]) * CONFIG["n_folds"]}")
print(f"  Threshold optimization: {CONFIG['optimize_threshold']}")

Config ready.
  Seeds: [42, 43, 44, 45, 46, 47, 48] (7 seeds)
  Total models: 35
  Threshold optimization: True


In [ ]:
def parse_fasta(path):
    seqs = []
    with open(path) as f:
        cur = ""
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if cur:
                    seqs.append(cur)
                cur = ""
            else:
                cur += line
        if cur:
            seqs.append(cur)
    return seqs


def parse_fasta_labeled(path):
    """Parse FASTA where headers contain Pos_ or Neg_ prefix."""
    pos_seqs, neg_seqs = [], []
    with open(path) as f:
        cur_seq, cur_label = "", None
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if cur_seq and cur_label is not None:
                    if cur_label == 1:
                        pos_seqs.append(cur_seq)
                    else:
                        neg_seqs.append(cur_seq)
                cur_seq = ""
                if ">Pos_" in line or ">pos_" in line:
                    cur_label = 1
                else:
                    cur_label = 0
            else:
                cur_seq += line
        if cur_seq and cur_label is not None:
            if cur_label == 1:
                pos_seqs.append(cur_seq)
            else:
                neg_seqs.append(cur_seq)
    return pos_seqs, neg_seqs


# ── Balanced datasets (same as v5) ──
data = Path(CONFIG["data_dir"])
pos_train = parse_fasta(data / "pos_train.fa")
neg_train = parse_fasta(data / "neg_train.fa")
pos_test  = parse_fasta(data / "pos_test.fa")
neg_test  = parse_fasta(data / "neg_test.fa")

train_seqs   = pos_train + neg_train
train_labels = np.array([1]*len(pos_train) + [0]*len(neg_train))
test_seqs    = pos_test + neg_test
test_labels  = np.array([1]*len(pos_test) + [0]*len(neg_test))

print(f"Train: {len(pos_train)} pos + {len(neg_train)} neg = {len(train_seqs)}")
print(f"Balanced test: {len(pos_test)} pos + {len(neg_test)} neg = {len(test_seqs)}")

# ── Imbalanced test set ──
imb_pos, imb_neg = parse_fasta_labeled(CONFIG["imbalanced_file"])
imb_seqs   = imb_pos + imb_neg
imb_labels = np.array([1]*len(imb_pos) + [0]*len(imb_neg))
print(f"Imbalanced test: {len(imb_pos)} pos + {len(imb_neg)} neg = {len(imb_seqs)}")

# Sanity checks
for tag, seqs in [("train", train_seqs), ("balanced test", test_seqs), ("imbalanced test", imb_seqs)]:
    assert all(len(s) == 41 for s in seqs), f"{tag}: not all 41-mers"
    assert all(s[20] == 'T' for s in seqs), f"{tag}: not all center=T"
    print(f"  {tag}: all 41-mers, all center=T OK")

Train: 879 pos + 878 neg = 1757
Balanced test: 200 pos + 200 neg = 400
Imbalanced test: 182 pos + 991 neg = 1173
  train: all 41-mers, all center=T OK
  balanced test: all 41-mers, all center=T OK
  imbalanced test: all 41-mers, all center=T OK


In [ ]:
AA = "ACDEFGHIKLMNPQRSTVWY"
AA2I = {a: i for i, a in enumerate(AA)}

BLOSUM62 = np.array([
    [ 4, 0,-2,-1,-2, 0,-2,-1,-1,-1,-1,-2,-1,-1,-1, 1, 0, 0,-3,-2],
    [ 0, 9,-3,-4,-2,-3,-3,-1,-3,-1,-1,-3,-3,-3,-3,-1,-1,-1,-2,-2],
    [-2,-3, 6, 2,-3,-1,-1,-3,-1,-4,-3, 1,-1, 0,-2, 0,-1,-3,-4,-3],
    [-1,-4, 2, 5,-3,-2, 0,-3, 1,-3,-2, 0,-1, 2, 0, 0,-1,-2,-3,-2],
    [-2,-2,-3,-3, 6,-3,-1, 0,-3, 0, 0,-3,-4,-3,-3,-2,-2,-1, 1, 3],
    [ 0,-3,-1,-2,-3, 6,-2,-4,-2,-4,-3, 0,-2,-2,-2, 0,-2,-3,-2,-3],
    [-2,-3,-1, 0,-1,-2, 8,-3,-1,-3,-2, 1,-2, 0, 0,-1,-2,-3,-2, 2],
    [-1,-1,-3,-3, 0,-4,-3, 4,-3, 2, 1,-3,-3,-3,-3,-2,-1, 3,-3,-1],
    [-1,-3,-1, 1,-3,-2,-1,-3, 5,-2,-1, 0,-1, 1, 2, 0,-1,-2,-3,-2],
    [-1,-1,-4,-3, 0,-4,-3, 2,-2, 4, 2,-3,-3,-2,-2,-2,-1, 1,-2,-1],
    [-1,-1,-3,-2, 0,-3,-2, 1,-1, 2, 5,-2,-2, 0,-1,-1,-1, 1,-1,-1],
    [-2,-3, 1, 0,-3, 0, 1,-3, 0,-3,-2, 6,-2, 0, 0, 1, 0,-3,-4,-2],
    [-1,-3,-1,-1,-4,-2,-2,-3,-1,-3,-2,-2, 7,-1,-2,-1,-1,-2,-4,-3],
    [-1,-3, 0, 2,-3,-2, 0,-3, 1,-2, 0, 0,-1, 5, 1, 0,-1,-2,-2,-1],
    [-1,-3,-2, 0,-3,-2, 0,-3, 2,-2,-1, 0,-2, 1, 5,-1,-1,-3,-3,-2],
    [ 1,-1, 0, 0,-2, 0,-1,-2, 0,-2,-1, 1,-1, 0,-1, 4, 1,-2,-3,-2],
    [ 0,-1,-1,-1,-2,-2,-2,-1,-1,-1,-1, 0,-1,-1,-1, 1, 5, 0,-2,-2],
    [ 0,-1,-3,-2,-1,-3,-3, 3,-2, 1, 1,-3,-2,-2,-3,-2, 0, 4,-3,-1],
    [-3,-2,-4,-3, 1,-2,-2,-3,-3,-2,-1,-4,-4,-2,-3,-3,-2,-3,11, 2],
    [-2,-2,-3,-2, 3,-3, 2,-1,-2,-1,-1,-2,-3,-1,-2,-2,-2,-1, 2, 7],
], dtype=np.float32)

def encode_seqs(seqs):
    N = len(seqs)
    blosum = np.zeros((N, 41, 20), dtype=np.float32)
    onehot = np.zeros((N, 41, 20), dtype=np.float32)
    for i, s in enumerate(seqs):
        for j, aa in enumerate(s):
            idx = AA2I[aa]
            blosum[i, j] = BLOSUM62[idx]
            onehot[i, j, idx] = 1.0
    return blosum, onehot

# Train + balanced test (same as v5)
train_blosum, train_onehot = encode_seqs(train_seqs)
test_blosum,  test_onehot  = encode_seqs(test_seqs)

b_mean = train_blosum.reshape(-1, 20).mean(axis=0)
b_std  = train_blosum.reshape(-1, 20).std(axis=0) + 1e-6
train_blosum_z = (train_blosum - b_mean) / b_std
test_blosum_z  = (test_blosum  - b_mean) / b_std

train_X_seq = np.concatenate([train_blosum_z, train_onehot], axis=-1)
test_X_seq  = np.concatenate([test_blosum_z,  test_onehot],  axis=-1)

# Imbalanced test (z-score with SAME train stats)
imb_blosum, imb_onehot = encode_seqs(imb_seqs)
imb_blosum_z = (imb_blosum - b_mean) / b_std
imb_X_seq = np.concatenate([imb_blosum_z, imb_onehot], axis=-1)

print(f"train_X_seq: {train_X_seq.shape}")
print(f"test_X_seq:  {test_X_seq.shape}")
print(f"imb_X_seq:   {imb_X_seq.shape}")

train_X_seq: (1757, 41, 40)
test_X_seq:  (400, 41, 40)
imb_X_seq:   (1173, 41, 40)


In [ ]:
print("Loading ESM-C 600M ...")
t0 = time.time()

plm = AutoModel.from_pretrained(
    CONFIG["plm_name"], trust_remote_code=True
).to(DEVICE).eval()

tokenizer = plm.tokenizer
for p in plm.parameters():
    p.requires_grad = False

print(f"  loaded in {time.time()-t0:.1f}s")

@torch.no_grad()
def extract_per_residue(seqs, batch_size=32):
    all_embeds = []
    for i in range(0, len(seqs), batch_size):
        chunk = seqs[i:i+batch_size]
        tok = tokenizer(chunk, return_tensors="pt", padding=True)
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            out = plm(**tok)
        embeds = out.last_hidden_state[:, 1:CONFIG["seq_len"]+1, :]
        all_embeds.append(embeds.cpu().float().numpy())
    return np.concatenate(all_embeds, axis=0)

print("Extracting train ...")
train_X_plm = extract_per_residue(train_seqs)
print(f"  train: {train_X_plm.shape}")

print("Extracting balanced test ...")
test_X_plm = extract_per_residue(test_seqs)
print(f"  test:  {test_X_plm.shape}")

print("Extracting imbalanced test ...")
imb_X_plm = extract_per_residue(imb_seqs)
print(f"  imb:   {imb_X_plm.shape}")

del plm, tokenizer
torch.cuda.empty_cache()
print(f"Done. Total: {time.time()-t0:.1f}s")

Loading ESM-C 600M ...


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

[transformers] ESMplusplusModel LOAD REPORT from: Synthyra/ESMplusplus_large
Key                            | Status     |  | 
-------------------------------+------------+--+-
sequence_head.{0, 2, 3}.weight | UNEXPECTED |  | 
sequence_head.{0, 2, 3}.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  loaded in 0.7s
Extracting train ...
  train: (1757, 41, 1152)
Extracting balanced test ...
  test:  (400, 41, 1152)
Extracting imbalanced test ...
  imb:   (1173, 41, 1152)
Done. Total: 3.0s


In [ ]:
class DualBranchOTG(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg["proj_dim"]

        # Branch A: Transformer on ESM-C per-residue
        self.input_proj = nn.Sequential(
            nn.Linear(cfg["plm_dim"], d),
            nn.LayerNorm(d),
        )
        self.pos_embed = nn.Embedding(cfg["seq_len"], d)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d,
            nhead=cfg["n_heads"],
            dim_feedforward=d * cfg["ff_mult"],
            dropout=cfg["trans_dropout"],
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=cfg["n_transformer_layers"],
        )
        self.center_idx = cfg["center_idx"]

        # Branch B: CNN-BiGRU
        self.convs = nn.ModuleList([
            nn.Conv1d(cfg["seq_feat_dim"], cfg["conv_channels"], kernel_size=k, padding=k // 2)
            for k in cfg["conv_kernels"]
        ])
        self.gru = nn.GRU(
            input_size=cfg["conv_channels"],
            hidden_size=cfg["gru_hidden"],
            batch_first=True,
            bidirectional=True,
        )
        self.seq_proj = nn.Sequential(
            nn.Dropout(cfg["seq_dropout"]),
            nn.Linear(cfg["gru_hidden"] * 2, cfg["branch_b_out"]),
            nn.BatchNorm1d(cfg["branch_b_out"]),
            nn.LeakyReLU(),
        )

        # Fusion: concat -> MLP
        fused_dim = cfg["branch_a_out"] + cfg["branch_b_out"]
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Dropout(cfg["clf_dropout"]),
            nn.Linear(fused_dim, cfg["fused_hidden"]),
            nn.GELU(),
            nn.Dropout(cfg["clf_dropout"]),
            nn.Linear(cfg["fused_hidden"], 2),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.trunc_normal_(m.weight, std=0.02)

    def forward(self, x_plm, x_seq):
        # Branch A
        h = self.input_proj(x_plm)
        pos = torch.arange(h.size(1), device=h.device)
        h = h + self.pos_embed(pos)
        h = self.transformer(h)
        f_plm = h[:, self.center_idx, :]

        # Branch B
        x = x_seq.transpose(1, 2)
        cnn_out = [F.relu(conv(x)).max(dim=-1).values for conv in self.convs]
        f_cnn = torch.stack(cnn_out, dim=1)
        _, h_n = self.gru(f_cnn)
        f_gru = torch.cat([h_n[0], h_n[1]], dim=-1)
        f_seq = self.seq_proj(f_gru)

        # Fuse + classify
        fused = torch.cat([f_plm, f_seq], dim=-1)
        return self.classifier(fused)


_m = DualBranchOTG(CONFIG).to(DEVICE)
n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"Model params: {n_params:,} ({n_params/1e6:.2f}M)")
del _m

Model params: 2,095,746 (2.10M)


In [ ]:
class DualBranchDataset(Dataset):
    def __init__(self, X_plm, X_seq, y):
        self.X_plm = torch.from_numpy(X_plm).float()
        self.X_seq = torch.from_numpy(X_seq).float()
        self.y = torch.from_numpy(y).long()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X_plm[i], self.X_seq[i], self.y[i]


def compute_metrics(probs, y, thr=0.5):
    preds = (probs >= thr).astype(int)
    return {
        "MCC": matthews_corrcoef(y, preds),
        "ACC": accuracy_score(y, preds),
        "SEN": recall_score(y, preds, pos_label=1),
        "SPE": recall_score(y, preds, pos_label=0),
    }


@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for xp, xs, y in loader:
        logits = model(xp.to(DEVICE), xs.to(DEVICE))
        probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


def mixup_batch(xp, xs, y, alpha):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    lam = max(lam, 1 - lam)
    batch_size = xp.size(0)
    idx = torch.randperm(batch_size, device=xp.device)
    xp_mix = lam * xp + (1 - lam) * xp[idx]
    xs_mix = lam * xs + (1 - lam) * xs[idx]
    return xp_mix, xs_mix, y, y[idx], lam


def mixup_cross_entropy(logits, y_a, y_b, lam, smoothing=0.0):
    ce_fn = nn.CrossEntropyLoss(label_smoothing=smoothing)
    return lam * ce_fn(logits, y_a) + (1 - lam) * ce_fn(logits, y_b)


def swa_phase(model, train_loader, cfg):
    swa_lr_base = cfg["lr"] * cfg["swa_lr_factor"]
    swa_lr_max  = cfg["lr"]
    opt = torch.optim.SGD(model.parameters(), lr=swa_lr_base,
                          momentum=0.9, weight_decay=cfg["weight_decay"])
    snapshots = []
    n_swa = cfg["swa_epochs"]

    for ep in range(n_swa):
        t = ep / max(n_swa - 1, 1)
        lr = swa_lr_base + 0.5 * (swa_lr_max - swa_lr_base) * (1 + math.cos(math.pi * t))
        for pg in opt.param_groups:
            pg['lr'] = lr

        model.train()
        for xp, xs, y in train_loader:
            xp, xs, y = xp.to(DEVICE), xs.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = F.cross_entropy(model(xp, xs), y, label_smoothing=cfg["label_smoothing"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            opt.step()
        snapshots.append(copy.deepcopy(model.state_dict()))

    avg_state = {}
    for key in snapshots[0]:
        avg_state[key] = torch.mean(torch.stack([s[key].float() for s in snapshots]), dim=0)
    model.load_state_dict(avg_state)
    print(f"    SWA: averaged {len(snapshots)} snapshots")
    return model


def train_one_fold(model, train_loader, val_loader, cfg, fold_id):
    opt = torch.optim.AdamW(
        model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"]
    )
    best_mcc, best_state, wait = -2.0, None, 0

    for epoch in range(cfg["max_epochs"]):
        model.train()
        total_loss = 0.0
        for xp, xs, y in train_loader:
            xp, xs, y = xp.to(DEVICE), xs.to(DEVICE), y.to(DEVICE)
            xp_m, xs_m, y_a, y_b, lam = mixup_batch(xp, xs, y, cfg["mixup_alpha"])
            opt.zero_grad()
            logits = model(xp_m, xs_m)
            loss = mixup_cross_entropy(logits, y_a, y_b, lam, cfg["label_smoothing"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            opt.step()
            total_loss += loss.item() * y.size(0)
        avg_loss = total_loss / len(train_loader.dataset)

        val_probs, val_y = predict_probs(model, val_loader)
        val_mcc = matthews_corrcoef(val_y, (val_probs >= cfg["threshold"]).astype(int))

        improved = val_mcc > best_mcc
        if improved:
            best_mcc = val_mcc
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if epoch % 10 == 0 or improved:
            tag = " *" if improved else ""
            print(f"  f{fold_id} ep{epoch:3d} | loss {avg_loss:.4f} | val MCC {val_mcc:.4f}{tag}")

        if wait >= cfg["patience"]:
            print(f"  f{fold_id}: early stop ep{epoch} | best val MCC {best_mcc:.4f}")
            break

    model.load_state_dict(best_state)
    model = swa_phase(model, train_loader, cfg)

    swa_probs, swa_y = predict_probs(model, val_loader)
    swa_mcc = matthews_corrcoef(swa_y, (swa_probs >= cfg["threshold"]).astype(int))
    print(f"  f{fold_id}: SWA val MCC {swa_mcc:.4f} (was {best_mcc:.4f}, delta {swa_mcc-best_mcc:+.4f})")

    if swa_mcc >= best_mcc:
        best_mcc = swa_mcc
        print(f"    -> keeping SWA weights")
    else:
        model.load_state_dict(best_state)
        print(f"    -> reverting to pre-SWA weights")

    return model, best_mcc

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def run_experiment(train_X_plm, train_X_seq, train_labels,
                   test_X_plm, test_X_seq, test_labels,
                   imb_X_plm, imb_X_seq, imb_labels, cfg, seed):
    set_seed(seed)
    print(f"\n{'='*60}")
    print(f"  SEED {seed}")
    print(f"{'='*60}")

    skf = StratifiedKFold(
        n_splits=cfg["n_folds"], shuffle=True, random_state=seed
    )

    fold_states = []
    fold_val_mccs = []
    all_val_probs = []  # for threshold optimization
    all_val_labels = []
    t0 = time.time()

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_X_plm, train_labels)):
        print(f"\n--- Fold {fold+1}/{cfg['n_folds']} ---")
        tr_ds = DualBranchDataset(train_X_plm[tr_idx], train_X_seq[tr_idx], train_labels[tr_idx])
        va_ds = DualBranchDataset(train_X_plm[va_idx], train_X_seq[va_idx], train_labels[va_idx])
        tr_ld = DataLoader(tr_ds, batch_size=cfg["batch_size"], shuffle=True)
        va_ld = DataLoader(va_ds, batch_size=cfg["batch_size"], shuffle=False)

        model = DualBranchOTG(cfg).to(DEVICE)
        model, best_val_mcc = train_one_fold(model, tr_ld, va_ld, cfg, fold+1)
        fold_states.append(copy.deepcopy(model.state_dict()))
        fold_val_mccs.append(best_val_mcc)

        # Collect val probs for threshold search
        vp, vy = predict_probs(model, va_ld)
        all_val_probs.append(vp)
        all_val_labels.append(vy)

    print(f"\nCV val MCCs: {[f'{m:.4f}' for m in fold_val_mccs]}")
    print(f"Mean: {np.mean(fold_val_mccs):.4f} +/- {np.std(fold_val_mccs):.4f}")
    print(f"Time: {(time.time()-t0)/60:.1f} min")

    # ── Balanced test ensemble ──
    test_ds = DualBranchDataset(test_X_plm, test_X_seq, test_labels)
    test_ld = DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False)

    fold_probs_bal = []
    for sd in fold_states:
        m = DualBranchOTG(cfg).to(DEVICE)
        m.load_state_dict(sd)
        probs, _ = predict_probs(m, test_ld)
        fold_probs_bal.append(probs)

    ens_bal = np.mean(np.stack(fold_probs_bal), axis=0)

    # ── Imbalanced test ensemble ──
    imb_ds = DualBranchDataset(imb_X_plm, imb_X_seq, imb_labels)
    imb_ld = DataLoader(imb_ds, batch_size=cfg["batch_size"], shuffle=False)

    fold_probs_imb = []
    for sd in fold_states:
        m = DualBranchOTG(cfg).to(DEVICE)
        m.load_state_dict(sd)
        probs, _ = predict_probs(m, imb_ld)
        fold_probs_imb.append(probs)

    ens_imb = np.mean(np.stack(fold_probs_imb), axis=0)

    bal_metrics = compute_metrics(ens_bal, test_labels, cfg["threshold"])
    imb_metrics = compute_metrics(ens_imb, imb_labels, cfg["threshold"])

    print("\n5-fold ensemble (balanced):")
    for k, v in bal_metrics.items():
        print(f"  {k}: {v:.4f}")
    print("5-fold ensemble (imbalanced):")
    for k, v in imb_metrics.items():
        print(f"  {k}: {v:.4f}")

    return {
        "seed": seed,
        "fold_val_mccs": fold_val_mccs,
        "fold_probs_bal": fold_probs_bal,
        "fold_probs_imb": fold_probs_imb,
        "val_probs": all_val_probs,
        "val_labels": all_val_labels,
        "ens5_bal": {k: float(v) for k, v in bal_metrics.items()},
        "ens5_imb": {k: float(v) for k, v in imb_metrics.items()},
    }

In [ ]:
all_results = []
all_probs_bal = []
all_probs_imb = []
all_val_probs_flat = []
all_val_labels_flat = []

for seed in CONFIG["seeds"]:
    r = run_experiment(train_X_plm, train_X_seq, train_labels,
                       test_X_plm, test_X_seq, test_labels,
                       imb_X_plm, imb_X_seq, imb_labels, CONFIG, seed)
    all_results.append(r)
    all_probs_bal.extend(r['fold_probs_bal'])
    all_probs_imb.extend(r['fold_probs_imb'])
    all_val_probs_flat.extend(r['val_probs'])
    all_val_labels_flat.extend(r['val_labels'])

# Mega-ensemble probabilities
mega_bal = np.mean(np.stack(all_probs_bal), axis=0)
mega_imb = np.mean(np.stack(all_probs_imb), axis=0)

# Pool all validation probs for threshold optimization
pooled_val_probs  = np.concatenate(all_val_probs_flat)
pooled_val_labels = np.concatenate(all_val_labels_flat)
print(f"Pooled validation samples for threshold search: {len(pooled_val_probs)}")


  SEED 42

--- Fold 1/5 ---
  f1 ep  0 | loss 0.5474 | val MCC 0.6876 *
  f1 ep  1 | loss 0.4925 | val MCC 0.7189 *
  f1 ep  3 | loss 0.4460 | val MCC 0.7236 *
  f1 ep  7 | loss 0.3966 | val MCC 0.7504 *
  f1 ep 10 | loss 0.3755 | val MCC 0.7502
  f1 ep 15 | loss 0.3307 | val MCC 0.7898 *
  f1 ep 20 | loss 0.3264 | val MCC 0.7303
  f1: early stop ep25 | best val MCC 0.7898
    SWA: averaged 10 snapshots
  f1: SWA val MCC 0.7955 (was 0.7898, delta +0.0057)
    -> keeping SWA weights

--- Fold 2/5 ---
  f2 ep  0 | loss 0.5758 | val MCC 0.7105 *
  f2 ep  1 | loss 0.4892 | val MCC 0.7136 *
  f2 ep  2 | loss 0.5022 | val MCC 0.7296 *
  f2 ep  3 | loss 0.4453 | val MCC 0.7417 *
  f2 ep  6 | loss 0.4299 | val MCC 0.7592 *
  f2 ep 10 | loss 0.3257 | val MCC 0.6574
  f2 ep 15 | loss 0.3262 | val MCC 0.7654 *
  f2 ep 17 | loss 0.2792 | val MCC 0.7674 *
  f2 ep 20 | loss 0.2907 | val MCC 0.7626
  f2: early stop ep27 | best val MCC 0.7674
    SWA: averaged 10 snapshots
  f2: SWA val MCC 0.7614 (w

In [ ]:
def find_best_threshold(probs, labels, base_thr=0.5):
    """Find threshold that maximizes accuracy while keeping MCC >= MCC@base_thr."""
    base_mcc = matthews_corrcoef(labels, (probs >= base_thr).astype(int))
    base_acc = accuracy_score(labels, (probs >= base_thr).astype(int))
    print(f"  Base threshold {base_thr}: MCC={base_mcc:.4f}, ACC={base_acc:.4f}")

    best_thr, best_acc = base_thr, base_acc
    for thr in np.arange(0.30, 0.71, 0.005):
        preds = (probs >= thr).astype(int)
        mcc = matthews_corrcoef(labels, preds)
        acc = accuracy_score(labels, preds)
        if mcc >= base_mcc and acc > best_acc:
            best_acc = acc
            best_thr = thr

    print(f"  Optimized threshold {best_thr:.3f}: ACC={best_acc:.4f} (MCC still >= {base_mcc:.4f})")
    return best_thr


if CONFIG["optimize_threshold"]:
    print("Searching for accuracy-optimal threshold on pooled CV validation data...")
    opt_thr = find_best_threshold(pooled_val_probs, pooled_val_labels)
else:
    opt_thr = 0.5
    print("Using fixed threshold 0.5")

print(f"\nFinal threshold: {opt_thr:.3f}")

Searching for accuracy-optimal threshold on pooled CV validation data...
  Base threshold 0.5: MCC=0.7741, ACC=0.8869
  Optimized threshold 0.500: ACC=0.8869 (MCC still >= 0.7741)

Final threshold: 0.500


In [ ]:
n_models = len(all_probs_bal)

bal_05 = compute_metrics(mega_bal, test_labels, 0.5)
imb_05 = compute_metrics(mega_imb, imb_labels, 0.5)

# Per-seed stats (balanced)
per_seed_bal = {k: [] for k in ["MCC", "ACC", "SEN", "SPE"]}
for r in all_results:
    for k, v in r['ens5_bal'].items():
        per_seed_bal[k].append(v)

# Per-seed stats (imbalanced)
per_seed_imb = {k: [] for k in ["MCC", "ACC", "SEN", "SPE"]}
for r in all_results:
    for k, v in r['ens5_imb'].items():
        per_seed_imb[k].append(v)

# Best single seed (balanced)
best_idx_bal = np.argmax([r['ens5_bal']['MCC'] for r in all_results])
best_bal = all_results[best_idx_bal]

# Best single seed (imbalanced)
best_idx_imb = np.argmax([r['ens5_imb']['MCC'] for r in all_results])
best_imb = all_results[best_idx_imb]

DEEPOTG_BAL = {"MCC": 0.807, "ACC": 0.902, "SEN": 0.865, "SPE": 0.940}
DEEPOTG_IMB = {"MCC": 0.737, "ACC": 0.919, "SEN": 0.865, "SPE": 0.930}

print("=" * 75)
print("  BALANCED TEST SET")
print("=" * 75)
print(f"{'Method':<40} {'MCC':>8} {'ACC':>8} {'SEN':>8} {'SPE':>8}")
print("-" * 75)

# Ours: mean ± std
for k in ["MCC", "ACC", "SEN", "SPE"]:
    pass
mean_str = "  ".join(f"{np.mean(per_seed_bal[k]):.4f}" for k in ["MCC", "ACC", "SEN", "SPE"])
std_str  = "  ".join(f"±{np.std(per_seed_bal[k]):.4f}" for k in ["MCC", "ACC", "SEN", "SPE"])
print(f"{'Ours (mean ± std, 7 seeds)':<40} ", end="")
for k in ["MCC", "ACC", "SEN", "SPE"]:
    print(f"{np.mean(per_seed_bal[k]):>7.4f}±{np.std(per_seed_bal[k]):.3f}", end=" ")
print()

print(f"{'Ours (best seed ensemble)':<40} ", end="")
for k in ["MCC", "ACC", "SEN", "SPE"]:
    print(f"{best_bal['ens5_bal'][k]:>11.4f}", end=" ")
print()

print(f"{'DeepOTG (reported)':<40} ", end="")
for k in ["MCC", "ACC", "SEN", "SPE"]:
    print(f"{DEEPOTG_BAL[k]:>11.3f}", end=" ")
print()

print()
print("=" * 75)
print("  IMBALANCED TEST SET")
print("=" * 75)
print(f"{'Method':<40} {'MCC':>8} {'ACC':>8} {'SEN':>8} {'SPE':>8}")
print("-" * 75)

print(f"{'Ours (mean ± std, 7 seeds)':<40} ", end="")
for k in ["MCC", "ACC", "SEN", "SPE"]:
    print(f"{np.mean(per_seed_imb[k]):>7.4f}±{np.std(per_seed_imb[k]):.3f}", end=" ")
print()

print(f"{'Ours (best seed ensemble)':<40} ", end="")
for k in ["MCC", "ACC", "SEN", "SPE"]:
    print(f"{best_imb['ens5_imb'][k]:>11.4f}", end=" ")
print()

print(f"{'DeepOTG (reported)':<40} ", end="")
for k in ["MCC", "ACC", "SEN", "SPE"]:
    print(f"{DEEPOTG_IMB[k]:>11.3f}", end=" ")
print()

  BALANCED TEST SET
Method                                        MCC      ACC      SEN      SPE
---------------------------------------------------------------------------
Ours (mean ± std, 7 seeds)                0.8177±0.011  0.9079±0.005  0.8771±0.015  0.9386±0.020 
Ours (best seed ensemble)                     0.8334      0.9150      0.8700      0.9600 
DeepOTG (reported)                             0.807       0.902       0.865       0.940 

  IMBALANCED TEST SET
Method                                        MCC      ACC      SEN      SPE
---------------------------------------------------------------------------
Ours (mean ± std, 7 seeds)                0.6907±0.027  0.9032±0.014  0.8650±0.017  0.9102±0.019 
Ours (best seed ensemble)                     0.7220      0.9182      0.8571      0.9294 
DeepOTG (reported)                             0.737       0.919       0.865       0.930 


In [ ]:
results_path = Path(CONFIG["results_dir"]) / "results.json"
save_data = {
    "config": {k: v for k, v in CONFIG.items()},
    "per_seed": [{k: v for k, v in r.items()
                  if k not in ("fold_probs_bal", "fold_probs_imb", "val_probs", "val_labels")}
                 for r in all_results],
    "balanced": {
        "mega_ensemble": {k: float(v) for k, v in bal_05.items()},
        "per_seed_mean": {k: float(np.mean(per_seed_bal[k])) for k in per_seed_bal},
        "per_seed_std": {k: float(np.std(per_seed_bal[k])) for k in per_seed_bal},
        "best_seed": {"seed": best_bal["seed"], **best_bal["ens5_bal"]},
    },
    "imbalanced": {
        "mega_ensemble": {k: float(v) for k, v in imb_05.items()},
        "per_seed_mean": {k: float(np.mean(per_seed_imb[k])) for k in per_seed_imb},
        "per_seed_std": {k: float(np.std(per_seed_imb[k])) for k in per_seed_imb},
        "best_seed": {"seed": best_imb["seed"], **best_imb["ens5_imb"]},
    },
    "deepotg_reference": {"balanced": DEEPOTG_BAL, "imbalanced": DEEPOTG_IMB},
}
with open(results_path, "w") as f:
    json.dump(save_data, f, indent=2)
print(f"\nSaved to {results_path}")


Saved to results_v5_1_final/results.json
